In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from random import sample
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, r2_score, roc_auc_score, roc_curve, cohen_kappa_score, brier_score_loss
from sklearn.metrics import matthews_corrcoef

In [ ]:
#input of train data
data = pd.read_excel("/home/acdsd/Documents/TNBC/MACCS_FP_Trainset_DD.xlsx")
data

In [ ]:
df = pd.DataFrame(data)
df.reset_index(drop=True, inplace=True)
df

In [ ]:
df.Target.value_counts()

In [ ]:
y_train = df.Target
x_train = df.drop('Target', axis=1)

# TRAIN SET ACCURACY

In [ ]:
import xgboost as xgb
xgb = xgb.XGBClassifier(
    booster='gbtree',
    colsample_bylevel= 0.8666835869179276,
    colsample_bytree= 0.7640185157924907,
    learning_rate= 0.01878506337063723,
    min_child_weight= 2,
    gamma= 0.01699906275679294,
    subsample= 0.772048502917609,
    max_depth= 9,
    n_estimators= 716,
    tree_method= 'hist',
    scale_pos_weight= 4.32537877477197,
    alpha= 0.004524526412723003,
    reg_lambda= 0.00514376463018152,
    random_state= 79
)
xgb.fit(x_train, y_train)
acc_train=xgb.score(x_train,y_train)*100
print(acc_train)

In [ ]:
from sklearn.metrics import RocCurveDisplay
from numpy import interp
from sklearn.metrics import roc_curve,auc
cv = StratifiedKFold(n_splits=10,shuffle=False)
plt.figure(figsize=(7, 5), dpi=600)
x_train = x_train.T
fig1 = plt.figure(figsize=[12,12])
ax1 = fig1.add_subplot(111,aspect = 'equal')
tprs = []
aucs = []
mean_fpr = np.linspace(0,1,100)
i = 1
for train,test in cv.split(x_train,y_train):
    prediction = xgb.fit(x_train.iloc[train],y_train.iloc[train]).predict_proba(x_train.iloc[test])
    fpr, tpr, t = roc_curve(y_train.iloc[test], prediction[:, 1])
    tprs.append(interp(mean_fpr, fpr, tpr))
    roc_auc = auc(fpr, tpr)
    aucs.append(roc_auc)
    plt.plot(fpr, tpr, lw=2, alpha=0.3, label='ROC fold %d (AUC = %0.2f)' % (i, roc_auc))
    i= i+1

plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')
mean_tpr = np.mean(tprs, axis=0)
mean_auc = auc(mean_fpr, mean_tpr)
plt.plot(mean_fpr, mean_tpr, color='blue',
         label=r'Mean ROC (AUC = %0.2f )' % (mean_auc),lw=2, alpha=1)

plt.xlabel('False Positive Rate', fontsize = 20)
plt.ylabel('True Positive Rate', fontsize = 20)
plt.legend(loc="lower right")
plt.savefig('/home/acdsd/Documents/TNBC/XGBoost_MACCS_FP/CrossVal-XGB.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import pickle

# Save the model to a .pkl file
with open('/home/acdsd/Documents/TNBC/Saved_model_TNBC/XGB_MACCS_FP.pkl', 'wb') as file:
    pickle.dump(xgb, file)

print("Model saved to XGB_MACCS_FP.pkl")

In [ ]:
external_data = pd.read_excel("/home/acdsd/Documents/TNBC/IC50_compounds_1313/FingerPrint(FP)/MACCS/MACCS_fingerprints_1313.xlsx")
data_for_screening = pd.DataFrame(external_data)

In [ ]:
test_prob = xgb.predict_proba(data_for_screening)
test_prob_F = pd.DataFrame(test_prob)
test_prob_F.to_csv('/home/acdsd/Documents/TNBC/New_1313_XGB_MACCS_FP/XGB_probability_MACCSFP.csv')

# TESTING

In [ ]:
v_test_data = pd.read_excel("/home/acdsd/Documents/TNBC/MACCS_FP_Testset_DD.xlsx")
v_test_data = pd.DataFrame(v_test_data)
v_test_data.shape
v_test_data

In [ ]:
Y_v_test_data = v_test_data.Target
X_v_test_data = v_test_data.drop('Target', axis=1)

In [ ]:
# x_train = x_train.T
# y_train = y_train.T

In [ ]:
v_predictions = xgb.predict(X_v_test_data)
v_acc_test = xgb.score(X_v_test_data, Y_v_test_data)*100
print(v_acc_test)

In [ ]:
# performance metrics to calculate
from sklearn.metrics import matthews_corrcoef
print('Accuracy:%0.3f'% v_acc_test)
tn, fp, fn, tp = confusion_matrix(Y_v_test_data, v_predictions).ravel()
specificity = tn / (tn+fp)
sensitivity = tp / (tp+fn)
print('Sensitivity:%0.3f'% sensitivity)
print('Specificity:%0.3f'% specificity)
ba = 0.5 * (sensitivity + specificity)
print('Balance accuracy:%0.3f'% ba)
mcc = matthews_corrcoef(Y_v_test_data, v_predictions)
print('MCC: %0.3f'% mcc)
bloss = brier_score_loss(Y_v_test_data, v_predictions)
print('Brier_loss: %0.3f' % bloss)
FPR = fp/(fp+tn)
FNR = fn/(tp+fn)
Precision = tp/(tp+fp)
Recall = tp/(tp+fn)
print('False Postive rate: %0.3f' %FPR)
print('False Negative rate: %0.3f' %FNR)
print('Precision: %0.3f' %Precision)
print('Recall: %0.3f' %Recall)
f1=2*((Precision*Recall)/(Precision+Recall))
print('F1: %0.3f' %f1)
kappa = cohen_kappa_score(Y_v_test_data, v_predictions)
print('Kappa: %0.3f' %kappa)

In [ ]:
r_auc_score = roc_auc_score(Y_v_test_data, v_predictions)
print('AUC: %0.3f' % r_auc_score)

In [ ]:
# confusion matrix
cf =(confusion_matrix(Y_v_test_data, v_predictions))
print(classification_report(Y_v_test_data, v_predictions))

In [ ]:
cf_plt =sns.heatmap(cf,annot=True,cmap="Blues",fmt="d",cbar=False, annot_kws={"size": 24})
cf_plt.set(xlabel = "Predicted Value", ylabel ="True Value")
cf_plt

In [ ]:
fig = cf_plt.get_figure()
fig.savefig("/home/acdsd/Documents/TNBC/XGBoost_MACCS_FP/XGB_Con_mat.png")

In [ ]:
# ROC - AUC curve
r_probs = [0 for _ in range(len(Y_v_test_data))]
xgb_prob = xgb.predict_proba(X_v_test_data)
xgb_prob = xgb_prob[:,1]
#xgb_prob

In [ ]:
r_auc_score = roc_auc_score(Y_v_test_data, xgb_prob)
r_auc_score_1 = roc_auc_score(Y_v_test_data,r_probs)
print(r_auc_score)
fpr, tpr, _ = roc_curve(Y_v_test_data, xgb_prob)
rfpr, rtpr, _ = roc_curve(Y_v_test_data, r_probs)

In [ ]:
plt.figure(figsize=(7, 5), dpi=600)
plt.plot(fpr, tpr, marker='.', label='xgboost(AUC=%0.3f)' % r_auc_score)
plt.plot(rfpr, rtpr, marker='_' % r_auc_score_1)
plt.xlabel('False Positive Rate -->')
plt.ylabel('True Positive Rate -->')
plt.legend()
plt.savefig('/home/acdsd/Documents/TNBC/XGBoost_MACCS_FP/XGB_AUC.png', dpi=600, bbox_inches='tight') #to save the image
plt.show()

# External Validation


In [ ]:
e_test_data = pd.read_excel("external_test_set_486.xlsx")
e_test_data = pd.DataFrame(e_test_data)
e_test_data.shape

In [ ]:
Y_e_test_data = e_test_data.Target
X_e_test_data = e_test_data.drop('Target', axis=1)

In [ ]:
e_predictions = xgb.predict(X_e_test_data)
e_acc_test = xgb.score(X_e_test_data, Y_e_test_data)*100
print(e_acc_test)

In [ ]:
# performance metrics to calculate
from sklearn.metrics import matthews_corrcoef
print('Accuracy:%0.3f'% e_acc_test)
tn, fp, fn, tp = confusion_matrix(Y_e_test_data, e_predictions).ravel()
specificity = tn / (tn+fp)
sensitivity = tp / (tp+fn)
print('Sensitivity:%0.3f'% sensitivity)
print('Specificity:%0.3f'% specificity)
ba = 0.5 * (sensitivity + specificity)
print('Balance accuracy:%0.3f'% ba)
mcc = matthews_corrcoef(Y_e_test_data, e_predictions)
print('MCC: %0.3f'% mcc)
bloss = brier_score_loss(Y_e_test_data, e_predictions)
print('Brier_loss: %0.3f' % bloss)
FPR = fp/(fp+tn)
FNR = fn/(tp+fn)
Precision = tp/(tp+fp)
Recall = tp/(tp+fn)
print('False Postive rate: %0.3f' %FPR)
print('False Negative rate: %0.3f' %FNR)
print('Precision: %0.3f' %Precision)
print('Recall: %0.3f' %Recall)
f1=2*((Precision*Recall)/(Precision+Recall))
print('F1: %0.3f' %f1)
kappa = cohen_kappa_score(Y_e_test_data, e_predictions)
print('Kappa: %0.3f' %kappa)

In [ ]:
m2= xgb.predict_proba(X_e_test_data)
m2 = pd.DataFrame(m2)
m2.to_csv("xgb_Probabs.csv")

In [ ]:
m2_probs = xgb.predict_proba(X_e_test_data)
m2_binary = (m2_probs[:, 1] > 0.5).astype(int)
m2_binary_df = pd.DataFrame(m2_binary, columns=['Predicted_Class'])
m2_binary_df.to_csv("xgb_Probabs_Binary.csv", index=False)

# interpretation

In [ ]:
import shap
import matplotlib.pyplot as plt

def model_predict(x_train):
    return xgb.predict(x_train)

masker = shap.maskers.Independent(X_v_test_data)

explainer = shap.Explainer(model_predict, masker)

shap_values = explainer(X_v_test_data)

# Plot and save the SHAP summary plot
shap.summary_plot(shap_values, X_v_test_data, show=False)

# Save the plot to a file
plt.savefig('XGB_shap_summary_plot.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Convert SHAP values to a DataFrame
shap_values_df = pd.DataFrame(shap_values.values, columns=X_v_test_data.columns)

# Save the SHAP values DataFrame to a CSV file
shap_values_df.to_csv('XGB_shap_values.csv', index=False)